# Comprehensive Model Evaluation, Comparison, and Ablation Analysis

This notebook evaluates prediction artifacts from traditional baselines, single-modality models, ablation variants, and the main multimodal CNN-BiLSTM attention model across all six development folds and the official benchmark test set in accordance with SPEC.md and AGENTS.md.


### Resolve project paths portably


In [1]:
from pathlib import Path
import os
import sys

def resolve_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "customer_handover",
        Path.cwd().parent,
        Path.cwd().parent / "customer_handover"
    ]
    for c in candidates:
        if (c / "training_config.yaml").exists():
            return c.resolve()
    return Path.cwd().resolve()

BASE_DIR = resolve_base_dir()
CONFIG_PATH = BASE_DIR / "training_config.yaml"
PREDICTIONS_DIR = BASE_DIR / "predictions"
REPORTS_DIR = BASE_DIR / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def to_portable_path(p: Path) -> str:
    try:
        rel = p.relative_to(BASE_DIR)
        return f"<PROJECT_ROOT>/customer_handover/{rel.as_posix()}" if str(rel) != "." else "<PROJECT_ROOT>/customer_handover"
    except Exception:
        return f"<PROJECT_ROOT>/{p.name}"

print(f"Base Directory: {to_portable_path(BASE_DIR)}")
print(f"Predictions Directory: {to_portable_path(PREDICTIONS_DIR)}")
print(f"Reports Directory: {to_portable_path(REPORTS_DIR)}")
print(f"Plots Directory: {to_portable_path(PLOTS_DIR)}")


Base Directory: <PROJECT_ROOT>/customer_handover
Predictions Directory: <PROJECT_ROOT>/customer_handover/predictions
Reports Directory: <PROJECT_ROOT>/customer_handover/reports
Plots Directory: <PROJECT_ROOT>/customer_handover/reports/plots


### Load training and evaluation configuration


In [2]:
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

REQUIRED_SCHEMA = config.get("prediction_schema", {}).get("required_columns", [])
EVAL_METRICS = config.get("evaluation_metrics", {})
GROUND_TRUTH_WARNING = config.get("ground_truth", {}).get("bearing1_4_warning", "")

print(f"Required Schema Columns ({len(REQUIRED_SCHEMA)}): {REQUIRED_SCHEMA}")
print(f"Evaluation Metrics: {EVAL_METRICS}")
print(f"Ground Truth Caveat: {GROUND_TRUTH_WARNING}")


Required Schema Columns (14): ['run_id', 'model_name', 'fold', 'split', 'seed', 'bearing_id', 'condition_id', 'endpoint_acquisition_index', 'endpoint_timestamp', 'actual_rul_seconds', 'predicted_rul_seconds', 'absolute_error_seconds', 'is_temperature_fully_missing', 'is_final_endpoint']
Evaluation Metrics: {'window_level': ['mae_seconds', 'rmse_seconds'], 'bearing_endpoint': ['mae_seconds', 'rmse_seconds', 'ieee_phm_2012_score']}
Ground Truth Caveat: Bearing1_4 configured RUL is 339s. Known Full_Test_Set endpoint discrepancy must be reported separately and not silently overwritten.


### Inspect model selection manifest, environment, and run manifests


In [3]:
import json

model_sel_path = REPORTS_DIR / "model_selection.json"
env_path = REPORTS_DIR / "environment.json"
manifests_path = REPORTS_DIR / "run_manifests.json"

assert model_sel_path.exists(), "Model selection manifest not found! 03_deep_learning_training.ipynb must be run first."
assert env_path.exists(), "Environment manifest not found!"
assert manifests_path.exists(), "Run manifests not found!"

with open(model_sel_path, "r", encoding="utf-8") as f:
    model_sel = json.load(f)
with open(env_path, "r", encoding="utf-8") as f:
    env_info = json.load(f)
with open(manifests_path, "r", encoding="utf-8") as f:
    run_manifests = json.load(f)

print(f"Selection Protocol: {model_sel.get('selection_protocol')}")
print(f"Empirically Best Development Model: {model_sel.get('empirically_best_validation_model')}")
print(f"Predetermined Target Model: {model_sel.get('predetermined_target_model', {}).get('model_name')}")
print(f"Frozen Official Epoch Budget: {model_sel.get('frozen_official_configuration', {}).get('frozen_epoch_budget')} epochs")
print(f"Environment: {env_info.get('os_name')}, PyTorch {env_info.get('packages', {}).get('torch')}")
print(f"Total Consolidated Run Manifests: {len(run_manifests)} entries verified.")


Selection Protocol: development_validation_only
Empirically Best Development Model: {'model_name': 'LateConcatBaseline', 'endpoint_mae_s': 1326.4, 'endpoint_rmse_s': 1960.6, 'ieee_phm_score': None}
Predetermined Target Model: CNNBiLSTMAttentionRUL
Frozen Official Epoch Budget: 5 epochs
Environment: Windows 10, PyTorch 2.14.1+cpu
Total Consolidated Run Manifests: 75 entries verified.


### Implement canonical IEEE PHM 2012 challenge score metric


In [4]:
import numpy as np

def compute_canonical_ieee_phm_score(actual_rul: np.ndarray, pred_rul: np.ndarray):
    """
    Canonical IEEE PHM 2012 Prognostics Challenge score function:
    Defined ONLY when all actual RUL values are strictly positive (> 0).
    For each endpoint i:
      Er_i = 100 * (actual_i - pred_i) / actual_i
      if Er_i <= 0 (overestimation/early): A_i = exp(-ln(0.5) * Er_i / 5)
      if Er_i > 0 (underestimation/late):  A_i = exp(ln(0.5) * Er_i / 20)
    When actual RUL is 0 (as in run-to-failure failure endpoints), percentage-error denominator
    is zero so metric is mathematically undefined. Returns None (displayed as N/A).
    """
    if len(actual_rul) == 0:
        return None
    actual_arr = np.asarray(actual_rul, dtype=float)
    pred_arr = np.asarray(pred_rul, dtype=float)
    if np.any(actual_arr <= 0.0):
        return None
        
    accuracies = []
    for act_f, pred_f in zip(actual_arr, pred_arr):
        er = 100.0 * (act_f - pred_f) / act_f
        if er <= 0.0:
            acc = float(np.exp(-np.log(0.5) * er / 5.0))
        else:
            acc = float(np.exp(np.log(0.5) * er / 20.0))
        accuracies.append(min(1.0, max(0.0, acc)))
    return float(np.mean(accuracies))

# Unit verification with hand-calculated fixture (positive actual RUL)
fixture_act = np.array([1000.0, 1000.0, 1000.0])
fixture_pred = np.array([1000.0, 980.0, 1005.0])
fixture_score = compute_canonical_ieee_phm_score(fixture_act, fixture_pred)
print(f"Canonical Scorer Fixture Check (Positive Actual): Score = {fixture_score:.4f} (Expected ~0.9554) -> PASS")
zero_fixture_score = compute_canonical_ieee_phm_score(np.array([0.0, 1000.0]), np.array([100.0, 1000.0]))
assert zero_fixture_score is None, "Score must be None when actual RUL contains 0.0!"
print("Zero Actual RUL Denominator Check: Score = None (N/A) -> PASS")


Canonical Scorer Fixture Check (Positive Actual): Score = 0.9554 (Expected ~0.9554) -> PASS
Zero Actual RUL Denominator Check: Score = None (N/A) -> PASS


### Implement window and endpoint evaluation functions


In [5]:
import pandas as pd

def evaluate_predictions_dataframe(df: pd.DataFrame, model_name: str, subgroup_name: str = "All Bearings") -> dict:
    actual = df["actual_rul_seconds"].to_numpy()
    pred = df["predicted_rul_seconds"].to_numpy()
    
    # Window-level metrics
    w_mae = float(np.mean(np.abs(actual - pred)))
    w_rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
    
    # Endpoint-level metrics (1 row per bearing)
    endpoints = df[df["is_final_endpoint"]].copy()
    if len(endpoints) == 0:
        endpoints = df.groupby("bearing_id").last().reset_index()
        
    e_actual = endpoints["actual_rul_seconds"].to_numpy()
    e_pred = endpoints["predicted_rul_seconds"].to_numpy()
    
    e_mae = float(np.mean(np.abs(e_actual - e_pred)))
    e_rmse = float(np.sqrt(np.mean((e_actual - e_pred) ** 2)))
    phm_score = compute_canonical_ieee_phm_score(e_actual, e_pred)
    
    return {
        "Model": model_name,
        "Subgroup": subgroup_name,
        "Num Windows": len(df),
        "Num Endpoints": len(endpoints),
        "Window MAE (s)": round(w_mae, 1),
        "Window RMSE (s)": round(w_rmse, 1),
        "Endpoint MAE (s)": round(e_mae, 1),
        "Endpoint RMSE (s)": round(e_rmse, 1),
        "IEEE PHM Score": round(phm_score, 5) if phm_score is not None else None
    }

def format_table_display(df_in: pd.DataFrame) -> pd.DataFrame:
    df_disp = df_in.copy()
    if "IEEE PHM Score" in df_disp.columns:
        df_disp["IEEE PHM Score"] = df_disp["IEEE PHM Score"].apply(
            lambda v: "N/A" if v is None or pd.isna(v) else f"{v:.5f}"
        )
    return df_disp

print("Evaluation functions defined.")


Evaluation functions defined.


### Load development predictions from traditional and Deep Learning models across all 6 folds


In [6]:
df_naive = pd.read_csv(PREDICTIONS_DIR / "naive_baseline_dev_predictions.csv")
df_rf = pd.read_csv(PREDICTIONS_DIR / "rf_baseline_dev_predictions.csv")
df_dl = pd.read_csv(PREDICTIONS_DIR / "deep_learning_dev_predictions.csv")

print(f"Naive Baseline Predictions: {len(df_naive)} rows across {df_naive['fold'].nunique()} folds")
print(f"Random Forest Predictions: {len(df_rf)} rows across {df_rf['fold'].nunique()} folds")
print(f"Deep Learning Predictions: {len(df_dl)} rows across {df_dl['fold'].nunique()} folds, Models={df_dl['model_name'].unique().tolist()}")


Naive Baseline Predictions: 468 rows across 6 folds
Random Forest Predictions: 468 rows across 6 folds
Deep Learning Predictions: 2808 rows across 6 folds, Models=['CNNBiLSTMAttentionRUL', 'MainModel_NoAttention', 'MainModel_NoModalityDropout', 'VibrationOnlyBaseline', 'TemperatureOnlyBaseline', 'LateConcatBaseline']


### Compile master development model comparison table across all 6 folds


In [7]:
comparison_records = []

# Traditional Baselines
comparison_records.append(evaluate_predictions_dataframe(df_naive, "Naive Median Baseline"))
comparison_records.append(evaluate_predictions_dataframe(df_rf, "Random Forest (Handcrafted)"))

# Deep Learning Models across all 6 folds
for m_name in ["CNNBiLSTMAttentionRUL", "MainModel_NoAttention", "MainModel_NoModalityDropout", "LateConcatBaseline", "TemperatureOnlyBaseline", "VibrationOnlyBaseline"]:
    sub = df_dl[df_dl["model_name"] == m_name]
    if len(sub) > 0:
        comparison_records.append(evaluate_predictions_dataframe(sub, m_name))

df_master_comparison = pd.DataFrame(comparison_records)
print("Master Model 6-Fold Development Comparison:")
print(format_table_display(df_master_comparison)[["Model", "Num Windows", "Num Endpoints", "Window MAE (s)", "Window RMSE (s)", "Endpoint MAE (s)", "Endpoint RMSE (s)", "IEEE PHM Score"]].to_string(index=False))

df_master_comparison.to_csv(REPORTS_DIR / "master_model_comparison.csv", index=False)


Master Model 6-Fold Development Comparison:
                      Model  Num Windows  Num Endpoints  Window MAE (s)  Window RMSE (s)  Endpoint MAE (s)  Endpoint RMSE (s) IEEE PHM Score
      Naive Median Baseline          468              6          6043.4           8241.8            6401.7             6460.8            N/A
Random Forest (Handcrafted)          468              6          6859.8           8930.6            2553.8             3197.3            N/A
      CNNBiLSTMAttentionRUL          468              6          5396.8           7929.6            2979.8             3190.2            N/A
      MainModel_NoAttention          468              6          5100.7           7680.1            2552.0             3072.2            N/A
MainModel_NoModalityDropout          468              6          5702.3           8191.5            4089.0             4783.9            N/A
         LateConcatBaseline          468              6          5248.4           7420.4            1326.4    

### Perform comprehensive modality ablation study across all 6 development folds


In [8]:
ablation_order = [
    "CNNBiLSTMAttentionRUL",
    "MainModel_NoAttention",
    "MainModel_NoModalityDropout",
    "VibrationOnlyBaseline",
    "TemperatureOnlyBaseline",
    "LateConcatBaseline"
]

ablation_records = []
for m in ablation_order:
    sub = df_dl[df_dl["model_name"] == m]
    if len(sub) > 0:
        ablation_records.append(evaluate_predictions_dataframe(sub, m, "Modality Ablation"))

df_ablation = pd.DataFrame(ablation_records)
print("Modality Ablation Study Across All 6 Folds:")
print(format_table_display(df_ablation)[["Model", "Window MAE (s)", "Window RMSE (s)", "Endpoint MAE (s)", "Endpoint RMSE (s)", "IEEE PHM Score"]].to_string(index=False))

df_ablation.to_csv(REPORTS_DIR / "modality_ablation_results.csv", index=False)


Modality Ablation Study Across All 6 Folds:
                      Model  Window MAE (s)  Window RMSE (s)  Endpoint MAE (s)  Endpoint RMSE (s) IEEE PHM Score
      CNNBiLSTMAttentionRUL          5396.8           7929.6            2979.8             3190.2            N/A
      MainModel_NoAttention          5100.7           7680.1            2552.0             3072.2            N/A
MainModel_NoModalityDropout          5702.3           8191.5            4089.0             4783.9            N/A
      VibrationOnlyBaseline          4964.8           6987.3            1738.0             1947.8            N/A
    TemperatureOnlyBaseline          5570.1           7740.8            5839.7             6011.6            N/A
         LateConcatBaseline          5248.4           7420.4            1326.4             1960.6            N/A


### Evaluate temperature availability subgroups on development set


In [9]:
subgroup_records = []

for m_label, df_model in [("Random Forest", df_rf), ("Main Model (CNNBiLSTMAttentionRUL)", df_dl[df_dl["model_name"] == "CNNBiLSTMAttentionRUL"])]:
    if len(df_model) == 0:
        continue
    # Subgroup 1: Available temperature
    df_avail = df_model[~df_model["is_temperature_fully_missing"]]
    if len(df_avail) > 0:
        subgroup_records.append(evaluate_predictions_dataframe(df_avail, m_label, "Usable Temperature"))
    # Subgroup 2: Fully missing temperature
    df_miss = df_model[df_model["is_temperature_fully_missing"]]
    if len(df_miss) > 0:
        subgroup_records.append(evaluate_predictions_dataframe(df_miss, m_label, "Fully Missing Temperature"))

df_subgroups = pd.DataFrame(subgroup_records)
print("Development Temperature Subgroup Comparison:")
print(format_table_display(df_subgroups)[["Model", "Subgroup", "Num Windows", "Endpoint MAE (s)", "Endpoint RMSE (s)", "IEEE PHM Score"]].to_string(index=False))

df_subgroups.to_csv(REPORTS_DIR / "subgroup_temperature_comparison.csv", index=False)


Development Temperature Subgroup Comparison:
                             Model                  Subgroup  Num Windows  Endpoint MAE (s)  Endpoint RMSE (s) IEEE PHM Score
                     Random Forest        Usable Temperature          317            2800.3             3599.2            N/A
                     Random Forest Fully Missing Temperature          151            2060.8             2181.9            N/A
Main Model (CNNBiLSTMAttentionRUL)        Usable Temperature          317            3330.5             3535.1            N/A
Main Model (CNNBiLSTMAttentionRUL) Fully Missing Temperature          151            2278.3             2353.4            N/A


### Plot actual versus predicted RUL trajectory across time


In [10]:
import matplotlib.pyplot as plt

b11_rf = df_rf[df_rf["bearing_id"] == "Bearing1_1"].sort_values("endpoint_acquisition_index")
b11_main = df_dl[(df_dl["model_name"] == "CNNBiLSTMAttentionRUL") & (df_dl["bearing_id"] == "Bearing1_1")].sort_values("endpoint_acquisition_index")

plt.figure(figsize=(10, 5))
if len(b11_rf) > 0:
    plt.plot(b11_rf["endpoint_acquisition_index"], b11_rf["actual_rul_seconds"], 'k-', label="Ground Truth RUL", linewidth=2)
    plt.plot(b11_rf["endpoint_acquisition_index"], b11_rf["predicted_rul_seconds"], 'b--', label="Random Forest Baseline", alpha=0.8)
if len(b11_main) > 0:
    plt.plot(b11_main["endpoint_acquisition_index"], b11_main["predicted_rul_seconds"], 'r-.', label="Main Model (CNN-BiLSTM-Attn)", alpha=0.8)

plt.xlabel("Acquisition Index")
plt.ylabel("Remaining Useful Life (seconds)")
plt.title("Bearing1_1: Actual vs Predicted RUL Trajectory Across Time")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

rul_curve_path = PLOTS_DIR / "rul_trajectory_bearing1_1.png"
plt.savefig(rul_curve_path, dpi=150)
plt.close()
print(f"RUL trajectory plot exported to: {to_portable_path(rul_curve_path)}")


RUL trajectory plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/rul_trajectory_bearing1_1.png


### Plot endpoint actual versus predicted scatter comparison across development folds


In [11]:
plt.figure(figsize=(8, 8))

for m_label, df in [("Random Forest", df_rf), ("CNNBiLSTMAttentionRUL", df_dl[df_dl["model_name"] == "CNNBiLSTMAttentionRUL"])]:
    if len(df) == 0:
        continue
    endpoints = df[df["is_final_endpoint"]]
    plt.scatter(endpoints["actual_rul_seconds"], endpoints["predicted_rul_seconds"], label=m_label, s=80, alpha=0.75)

max_val = max(df_rf["actual_rul_seconds"].max(), 8000)
plt.plot([0, max_val], [0, max_val], "k--", label="Ideal Prediction (y=x)", alpha=0.6)

plt.xlabel("Actual Endpoint RUL (seconds)")
plt.ylabel("Predicted Endpoint RUL (seconds)")
plt.title("Development Endpoints: Actual vs Predicted Scatter (All 6 Folds)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

scatter_path = PLOTS_DIR / "endpoint_scatter_comparison.png"
plt.savefig(scatter_path, dpi=150)
plt.close()
print(f"Scatter plot exported to: {to_portable_path(scatter_path)}")


Scatter plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/endpoint_scatter_comparison.png


### Plot prediction error distribution across models


In [12]:
plt.figure(figsize=(10, 5))

error_data = [
    df_naive["absolute_error_seconds"].to_numpy(),
    df_rf["absolute_error_seconds"].to_numpy()
]
labels = ["Naive Median", "Random Forest"]

for m_name, grp in df_dl.groupby("model_name"):
    error_data.append(grp["absolute_error_seconds"].to_numpy())
    labels.append(m_name)

plt.boxplot(error_data, tick_labels=labels, showfliers=False)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Absolute Error (seconds)")
plt.title("Window-Level Absolute Error Distribution by Model (6 Folds)")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

error_box_path = PLOTS_DIR / "error_distribution_boxplot.png"
plt.savefig(error_box_path, dpi=150)
plt.close()
print(f"Error distribution plot exported to: {to_portable_path(error_box_path)}")


Error distribution plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/error_distribution_boxplot.png


### Plot modality ablation bar chart comparison


In [13]:
fig, ax = plt.subplots(figsize=(10, 5))

models = df_ablation["Model"].tolist()
mae_vals = df_ablation["Endpoint MAE (s)"].tolist()
rmse_vals = df_ablation["Endpoint RMSE (s)"].tolist()

x = np.arange(len(models))
width = 0.35

rects1 = ax.bar(x - width/2, mae_vals, width, label='Endpoint MAE (s)', color='royalblue', alpha=0.85)
rects2 = ax.bar(x + width/2, rmse_vals, width, label='Endpoint RMSE (s)', color='salmon', alpha=0.85)
ax.set_ylabel('Error (seconds)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha="right")
ax.legend()
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.title("Modality Ablation Performance Comparison (All 6 Development Folds)")
fig.tight_layout()

ablation_plot_path = PLOTS_DIR / "modality_ablation_comparison.png"
plt.savefig(ablation_plot_path, dpi=150)
plt.close()
print(f"Ablation comparison plot exported to: {to_portable_path(ablation_plot_path)}")


Ablation comparison plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/modality_ablation_comparison.png


### Load official test predictions across multi-seed budget and evaluate benchmark metrics


In [14]:
df_official_dl = pd.read_csv(PREDICTIONS_DIR / "official_test_predictions.csv")
df_official_rf = pd.read_csv(PREDICTIONS_DIR / "rf_official_test_predictions.csv")

official_benchmarks = []
official_benchmarks.append(evaluate_predictions_dataframe(df_official_rf, "Random Forest (Handcrafted)", "Official Test Benchmark"))

# Evaluate each seed independently
for seed_val in [42, 123, 999]:
    grp = df_official_dl[df_official_dl["seed"] == seed_val]
    if len(grp) > 0:
        official_benchmarks.append(evaluate_predictions_dataframe(grp, f"CNNBiLSTMAttentionRUL (Seed {seed_val})", "Official Test Benchmark"))

# Multi-seed ensemble average
ensemble_dl = df_official_dl[df_official_dl["model_name"] == "CNNBiLSTMAttentionRUL_Ensemble"]
if len(ensemble_dl) == 0:
    ensemble_dl = df_official_dl[df_official_dl["seed"].isin([42, 123, 999])].groupby(["bearing_id", "endpoint_acquisition_index"]).agg({
        "actual_rul_seconds": "first",
        "predicted_rul_seconds": "mean",
        "is_temperature_fully_missing": "first",
        "is_final_endpoint": "first"
    }).reset_index()

official_benchmarks.append(evaluate_predictions_dataframe(ensemble_dl, "CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)", "Official Test Benchmark"))

df_official_eval = pd.DataFrame(official_benchmarks)
print("Official Test Benchmark Results across Seeds (11 Bearings):")
print(df_official_eval[["Model", "Num Endpoints", "Window MAE (s)", "Endpoint MAE (s)", "Endpoint RMSE (s)", "IEEE PHM Score"]].to_string(index=False))

df_official_eval.to_csv(REPORTS_DIR / "official_test_benchmark_results.csv", index=False)


Official Test Benchmark Results across Seeds (11 Bearings):
                                      Model  Num Endpoints  Window MAE (s)  Endpoint MAE (s)  Endpoint RMSE (s)  IEEE PHM Score
                Random Forest (Handcrafted)             11          6558.1            3502.9             4378.0         0.11422
            CNNBiLSTMAttentionRUL (Seed 42)             11          8450.9            3751.8             5093.6         0.13292
           CNNBiLSTMAttentionRUL (Seed 123)             11          8090.6            2972.5             3529.0         0.09949
           CNNBiLSTMAttentionRUL (Seed 999)             11          8447.1            3349.9             4018.0         0.08955
CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)             11          8191.5            3264.0             4140.0         0.08465


### Evaluate official test subgroups by temperature availability


In [15]:
official_subgroups = []

for m_label, df_m in [("Random Forest", df_official_rf), ("CNNBiLSTMAttentionRUL (Ensemble)", ensemble_dl)]:
    # Subgroup 1: Available temperature
    df_avail = df_m[~df_m["is_temperature_fully_missing"]]
    if len(df_avail) > 0:
        official_subgroups.append(evaluate_predictions_dataframe(df_avail, m_label, "Usable Temperature"))
    # Subgroup 2: Fully missing temperature
    df_miss = df_m[df_m["is_temperature_fully_missing"]]
    if len(df_miss) > 0:
        official_subgroups.append(evaluate_predictions_dataframe(df_miss, m_label, "Fully Missing Temperature"))

df_official_subgroups = pd.DataFrame(official_subgroups)
print("Official Test Temperature Subgroup Comparison:")
print(df_official_subgroups[["Model", "Subgroup", "Num Endpoints", "Endpoint MAE (s)", "Endpoint RMSE (s)", "IEEE PHM Score"]].to_string(index=False))

df_official_subgroups.to_csv(REPORTS_DIR / "subgroup_temperature_comparison.csv", index=False)


Official Test Temperature Subgroup Comparison:
                           Model                  Subgroup  Num Endpoints  Endpoint MAE (s)  Endpoint RMSE (s)  IEEE PHM Score
                   Random Forest        Usable Temperature              8            3677.2             4592.8         0.05001
                   Random Forest Fully Missing Temperature              3            3038.3             3745.6         0.28544
CNNBiLSTMAttentionRUL (Ensemble)        Usable Temperature              8            3364.7             4315.6         0.08592
CNNBiLSTMAttentionRUL (Ensemble) Fully Missing Temperature              3            2995.4             3630.5         0.08126


### Analyze and report Bearing1_4 ground truth caveat


In [16]:
b14_pred_dl = ensemble_dl[(ensemble_dl["bearing_id"] == "Bearing1_4") & (ensemble_dl["is_final_endpoint"])].iloc[0]
b14_pred_rf = df_official_rf[(df_official_rf["bearing_id"] == "Bearing1_4") & (df_official_rf["is_final_endpoint"])].iloc[0]

err_dl = abs(b14_pred_dl['actual_rul_seconds'] - b14_pred_dl['predicted_rul_seconds'])
err_rf = abs(b14_pred_rf['actual_rul_seconds'] - b14_pred_rf['predicted_rul_seconds'])

print("Bearing1_4 Ground Truth Caveat Analysis:")
print(f"  Configured Ground Truth (MDPI Table 2): {b14_pred_dl['actual_rul_seconds']:.1f} s")
print(f"  Random Forest Endpoint Prediction:      {b14_pred_rf['predicted_rul_seconds']:.1f} s (Absolute Error: {err_rf:.1f} s)")
print(f"  CNN-BiLSTM-Attention (Ensemble):         {b14_pred_dl['predicted_rul_seconds']:.1f} s (Absolute Error: {err_dl:.1f} s)")
print(f"  Caveat: {GROUND_TRUTH_WARNING}")


Bearing1_4 Ground Truth Caveat Analysis:
  Configured Ground Truth (MDPI Table 2): 339.0 s
  Random Forest Endpoint Prediction:      1011.3 s (Absolute Error: 672.3 s)
  CNN-BiLSTM-Attention (Ensemble):         2866.0 s (Absolute Error: 2527.0 s)
  Caveat: Bearing1_4 configured RUL is 339s. Known Full_Test_Set endpoint discrepancy must be reported separately and not silently overwritten.


### Plot actual versus predicted endpoints on official benchmark


In [17]:
plt.figure(figsize=(9, 8))

ep_rf = df_official_rf[df_official_rf["is_final_endpoint"]]
ep_dl = ensemble_dl[ensemble_dl["is_final_endpoint"]]

plt.scatter(ep_rf["actual_rul_seconds"], ep_rf["predicted_rul_seconds"], label="Random Forest Baseline", s=90, color="navy", alpha=0.8)
plt.scatter(ep_dl["actual_rul_seconds"], ep_dl["predicted_rul_seconds"], label="CNN-BiLSTM-Attention (Multi-Seed Ensemble)", s=90, color="crimson", alpha=0.8)

for _, row in ep_dl.iterrows():
    plt.annotate(row["bearing_id"], (row["actual_rul_seconds"], row["predicted_rul_seconds"]),
                 textcoords="offset points", xytext=(5, 5), fontsize=8, alpha=0.8)

max_benchmark = max(ep_rf["actual_rul_seconds"].max(), ep_dl["actual_rul_seconds"].max(), 8000)
plt.plot([0, max_benchmark], [0, max_benchmark], 'k--', label="Perfect Prediction (y=x)", alpha=0.6)

plt.xlabel("Actual Ground Truth Endpoint RUL (seconds)")
plt.ylabel("Predicted Endpoint RUL (seconds)")
plt.title("Official PHM 2012 Benchmark Test Set: Endpoint Predictions (11 Bearings)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

bench_plot_path = PLOTS_DIR / "official_test_endpoint_comparison.png"
plt.savefig(bench_plot_path, dpi=150)
plt.close()
print(f"Official benchmark comparison plot exported to: {to_portable_path(bench_plot_path)}")


Official benchmark comparison plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/official_test_endpoint_comparison.png


### Export comprehensive evaluation report and comparison tables


In [18]:
final_evaluation_report = {
    "evaluation_version": "2.1.0",
    "model_selection": model_sel,
    "development_6_fold_models": comparison_records,
    "temperature_subgroups_dev": subgroup_records,
    "modality_ablations_6_fold": ablation_records,
    "official_test_benchmarks": official_benchmarks,
    "official_temperature_subgroups": official_subgroups,
    "plots_generated": [
        str(rul_curve_path.name),
        str(scatter_path.name),
        str(error_box_path.name),
        str(ablation_plot_path.name),
        str(bench_plot_path.name)
    ],
    "ground_truth_caveats": [
        GROUND_TRUTH_WARNING
    ]
}

final_report_path = REPORTS_DIR / "final_evaluation_and_ablation_report.json"
with open(final_report_path, "w", encoding="utf-8") as f:
    json.dump(final_evaluation_report, f, indent=2)

print(f"Master evaluation report exported to: {to_portable_path(final_report_path)}")


Master evaluation report exported to: <PROJECT_ROOT>/customer_handover/reports/final_evaluation_and_ablation_report.json


### Compute and export per-fold development metrics across all models


In [19]:
fold_metric_records = []

# List of all 8 models to evaluate per fold
all_dev_models = [
    ("Naive Median Baseline", df_naive),
    ("Random Forest (Handcrafted)", df_rf)
]
for m_name in ["CNNBiLSTMAttentionRUL", "MainModel_NoAttention", "MainModel_NoModalityDropout", "LateConcatBaseline", "TemperatureOnlyBaseline", "VibrationOnlyBaseline"]:
    sub = df_dl[df_dl["model_name"] == m_name]
    if len(sub) > 0:
        all_dev_models.append((m_name, sub))

for m_label, df_m in all_dev_models:
    for f in sorted(df_m["fold"].unique()):
        sub_fold = df_m[df_m["fold"] == f]
        if len(sub_fold) == 0:
            continue
        actual = sub_fold["actual_rul_seconds"].to_numpy()
        pred = sub_fold["predicted_rul_seconds"].to_numpy()
        w_mae = float(np.mean(np.abs(actual - pred)))
        w_rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
        
        ep = sub_fold[sub_fold["is_final_endpoint"]]
        if len(ep) == 0:
            ep = sub_fold.groupby("bearing_id").last().reset_index()
        e_actual = ep["actual_rul_seconds"].to_numpy()
        e_pred = ep["predicted_rul_seconds"].to_numpy()
        e_mae = float(np.mean(np.abs(e_actual - e_pred)))
        e_rmse = float(np.sqrt(np.mean((e_actual - e_pred) ** 2)))
        
        fold_metric_records.append({
            "model_name": m_label,
            "fold": int(f),
            "bearing_id": ep.iloc[0]["bearing_id"] if len(ep) > 0 else f"Fold_{f}",
            "num_windows": len(sub_fold),
            "num_endpoints": len(ep),
            "window_mae_s": round(w_mae, 1),
            "window_rmse_s": round(w_rmse, 1),
            "endpoint_mae_s": round(e_mae, 1),
            "endpoint_rmse_s": round(e_rmse, 1),
            "ieee_phm_score": None  # Zero actual RUL at development failure endpoints
        })

df_fold_metrics = pd.DataFrame(fold_metric_records)
dev_fold_metrics_path = REPORTS_DIR / "development_fold_metrics.csv"
df_fold_metrics.to_csv(dev_fold_metrics_path, index=False)

print(f"Per-fold development metrics exported to: {to_portable_path(dev_fold_metrics_path)} ({len(df_fold_metrics)} records across 8 models)")
print(df_fold_metrics.head(12)[["model_name", "fold", "bearing_id", "endpoint_mae_s", "endpoint_rmse_s"]].to_string(index=False))


Per-fold development metrics exported to: <PROJECT_ROOT>/customer_handover/reports/development_fold_metrics.csv (48 records across 8 models)
                 model_name  fold bearing_id  endpoint_mae_s  endpoint_rmse_s
      Naive Median Baseline     0 Bearing1_1          4620.0           4620.0
      Naive Median Baseline     1 Bearing1_2          6930.0           6930.0
      Naive Median Baseline     2 Bearing2_1          6910.0           6910.0
      Naive Median Baseline     3 Bearing2_2          7050.0           7050.0
      Naive Median Baseline     4 Bearing3_1          6910.0           6910.0
      Naive Median Baseline     5 Bearing3_2          5990.0           5990.0
Random Forest (Handcrafted)     0 Bearing1_1           803.8            803.8
Random Forest (Handcrafted)     1 Bearing1_2          2492.0           2492.0
Random Forest (Handcrafted)     2 Bearing2_1          6571.1           6571.1
Random Forest (Handcrafted)     3 Bearing2_2          2777.5           2777.5
R

### Compute and export across-fold summary statistics with sample standard deviation


In [20]:
summary_records = []

for m_label, grp in df_fold_metrics.groupby("model_name"):
    w_maes = grp["window_mae_s"].to_numpy()
    w_rmses = grp["window_rmse_s"].to_numpy()
    e_maes = grp["endpoint_mae_s"].to_numpy()
    e_rmses = grp["endpoint_rmse_s"].to_numpy()
    
    # Calculate global concatenated row-level metrics for this model
    if m_label == "Naive Median Baseline":
        m_preds = df_naive
    elif m_label == "Random Forest (Handcrafted)":
        m_preds = df_rf
    else:
        m_preds = df_dl[df_dl["model_name"] == m_label]
        
    m_act_win = m_preds["actual_rul_seconds"].to_numpy()
    m_pred_win = m_preds["predicted_rul_seconds"].to_numpy()
    global_win_mae = round(float(np.mean(np.abs(m_pred_win - m_act_win))), 1)
    global_win_rmse = round(float(np.sqrt(np.mean((m_pred_win - m_act_win) ** 2))), 1)
    
    m_ep = m_preds[m_preds["is_final_endpoint"]]
    if len(m_ep) == 0:
        m_ep = m_preds.groupby("bearing_id").last().reset_index()
    m_act_ep = m_ep["actual_rul_seconds"].to_numpy()
    m_pred_ep = m_ep["predicted_rul_seconds"].to_numpy()
    global_ep_mae = round(float(np.mean(np.abs(m_pred_ep - m_act_ep))), 1)
    global_ep_rmse = round(float(np.sqrt(np.mean((m_pred_ep - m_act_ep) ** 2))), 1)

    summary_records.append({
        "model_name": m_label,
        "num_folds": len(grp),
        "window_mae_mean_s": round(float(np.mean(w_maes)), 1),
        "window_mae_std_s": round(float(np.std(w_maes, ddof=1)), 1),
        "window_mae_min_s": round(float(np.min(w_maes)), 1),
        "window_mae_max_s": round(float(np.max(w_maes)), 1),
        "window_rmse_mean_s": round(float(np.mean(w_rmses)), 1),
        "window_rmse_std_s": round(float(np.std(w_rmses, ddof=1)), 1),
        "window_rmse_min_s": round(float(np.min(w_rmses)), 1),
        "window_rmse_max_s": round(float(np.max(w_rmses)), 1),
        "endpoint_mae_mean_s": round(float(np.mean(e_maes)), 1),
        "endpoint_mae_std_s": round(float(np.std(e_maes, ddof=1)), 1),
        "endpoint_mae_min_s": round(float(np.min(e_maes)), 1),
        "endpoint_mae_max_s": round(float(np.max(e_maes)), 1),
        "mean_fold_endpoint_rmse_s": round(float(np.mean(e_rmses)), 1),
        "endpoint_rmse_mean_s": round(float(np.mean(e_rmses)), 1),
        "std_fold_endpoint_rmse_s": round(float(np.std(e_rmses, ddof=1)), 1),
        "endpoint_rmse_std_s": round(float(np.std(e_rmses, ddof=1)), 1),
        "endpoint_rmse_min_s": round(float(np.min(e_rmses)), 1),
        "endpoint_rmse_max_s": round(float(np.max(e_rmses)), 1),
        "global_window_mae_s": global_win_mae,
        "global_window_rmse_s": global_win_rmse,
        "global_endpoint_mae_s": global_ep_mae,
        "global_endpoint_rmse_s": global_ep_rmse,
        "std_convention": "sample_standard_deviation_ddof_1"
    })

df_fold_summary = pd.DataFrame(summary_records)
dev_fold_summary_path = REPORTS_DIR / "development_fold_summary.csv"
df_fold_summary.to_csv(dev_fold_summary_path, index=False)

print(f"Across-fold summary statistics exported to: {to_portable_path(dev_fold_summary_path)}")
print(df_fold_summary[["model_name", "endpoint_mae_mean_s", "mean_fold_endpoint_rmse_s", "global_endpoint_rmse_s"]].to_string(index=False))


Across-fold summary statistics exported to: <PROJECT_ROOT>/customer_handover/reports/development_fold_summary.csv
                 model_name  endpoint_mae_mean_s  mean_fold_endpoint_rmse_s  global_endpoint_rmse_s
      CNNBiLSTMAttentionRUL               2979.8                     2979.8                  3190.2
         LateConcatBaseline               1326.4                     1326.4                  1960.6
      MainModel_NoAttention               2552.0                     2552.0                  3072.2
MainModel_NoModalityDropout               4089.0                     4089.0                  4783.9
      Naive Median Baseline               6401.7                     6401.7                  6460.8
Random Forest (Handcrafted)               2553.8                     2553.8                  3197.3
    TemperatureOnlyBaseline               5839.7                     5839.7                  6011.6
      VibrationOnlyBaseline               1738.0                     1738.0           

### Compute and export official test endpoint diagnostics with signed and percentage errors


In [21]:
official_diag_records = []

official_model_sources = [
    ("Random Forest (Handcrafted)", df_official_rf),
    ("CNNBiLSTMAttentionRUL (Seed 42)", df_official_dl[df_official_dl["seed"] == 42]),
    ("CNNBiLSTMAttentionRUL (Seed 123)", df_official_dl[df_official_dl["seed"] == 123]),
    ("CNNBiLSTMAttentionRUL (Seed 999)", df_official_dl[df_official_dl["seed"] == 999]),
    ("CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)", ensemble_dl)
]

for m_label, df_src in official_model_sources:
    ep = df_src[df_src["is_final_endpoint"]].copy()
    for _, row in ep.iterrows():
        act = float(row["actual_rul_seconds"])
        pred = float(row["predicted_rul_seconds"])
        assert act > 0.0, f"Official test actual RUL must be positive! Found: {act}"
        signed_err = pred - act  # Convention: predicted - actual
        abs_err = abs(pred - act)
        ape = 100.0 * abs_err / act
        
        official_diag_records.append({
            "model_name": m_label,
            "bearing_id": row["bearing_id"],
            "condition_id": row["condition_id"],
            "actual_rul_seconds": round(act, 1),
            "predicted_rul_seconds": round(pred, 1),
            "signed_error_seconds": round(signed_err, 1),
            "absolute_error_seconds": round(abs_err, 1),
            "absolute_percentage_error": round(ape, 2),
            "temperature_subgroup": "Fully Missing Temperature" if row["is_temperature_fully_missing"] else "Usable Temperature",
            "within_10_percent": bool(ape <= 10.0),
            "within_20_percent": bool(ape <= 20.0)
        })

df_official_diag = pd.DataFrame(official_diag_records)
official_diag_path = REPORTS_DIR / "official_endpoint_diagnostics.csv"
df_official_diag.to_csv(official_diag_path, index=False)

print(f"Official endpoint diagnostics exported to: {to_portable_path(official_diag_path)} ({len(df_official_diag)} records)")
print(df_official_diag[df_official_diag['model_name'] == 'CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)'][['bearing_id', 'condition_id', 'actual_rul_seconds', 'predicted_rul_seconds', 'signed_error_seconds', 'absolute_percentage_error', 'within_20_percent']].to_string(index=False))


Official endpoint diagnostics exported to: <PROJECT_ROOT>/customer_handover/reports/official_endpoint_diagnostics.csv (55 records)
bearing_id condition_id  actual_rul_seconds  predicted_rul_seconds  signed_error_seconds  absolute_percentage_error  within_20_percent
Bearing1_3          OC1              5730.0                 2610.6               -3119.4                      54.44              False
Bearing1_4          OC1               339.0                 2866.0                2527.0                     745.43              False
Bearing1_5          OC1              1610.0                 1815.8                 205.8                      12.79               True
Bearing1_6          OC1              1460.0                 2978.1                1518.1                     103.98              False
Bearing1_7          OC1              7570.0                 2206.0               -5364.0                      70.86              False
Bearing2_3          OC2              7530.0                

### Compute and export official tolerance accuracy summaries at 10% and 20% thresholds


In [22]:
tolerance_records = []

for m_label, grp in df_official_diag.groupby("model_name"):
    n_endpoints = len(grp)
    w10_cnt = int(grp["within_10_percent"].sum())
    w20_cnt = int(grp["within_20_percent"].sum())
    
    tolerance_records.append({
        "model_name": m_label,
        "num_endpoints": n_endpoints,
        "within_10_percent_count": w10_cnt,
        "within_10_percent_rate": round(w10_cnt / n_endpoints, 4),
        "within_20_percent_count": w20_cnt,
        "within_20_percent_rate": round(w20_cnt / n_endpoints, 4)
    })

df_tolerance = pd.DataFrame(tolerance_records)
tolerance_path = REPORTS_DIR / "official_tolerance_summary.csv"
df_tolerance.to_csv(tolerance_path, index=False)

print(f"Official tolerance summary exported to: {to_portable_path(tolerance_path)}")
print(df_tolerance.to_string(index=False))


Official tolerance summary exported to: <PROJECT_ROOT>/customer_handover/reports/official_tolerance_summary.csv
                                 model_name  num_endpoints  within_10_percent_count  within_10_percent_rate  within_20_percent_count  within_20_percent_rate
CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)             11                        0                  0.0000                        1                  0.0909
           CNNBiLSTMAttentionRUL (Seed 123)             11                        0                  0.0000                        1                  0.0909
            CNNBiLSTMAttentionRUL (Seed 42)             11                        1                  0.0909                        2                  0.1818
           CNNBiLSTMAttentionRUL (Seed 999)             11                        0                  0.0000                        0                  0.0000
                Random Forest (Handcrafted)             11                        1                  0.

### Compute and export official multi-seed stability statistics


In [23]:
seed_model_names = [
    "CNNBiLSTMAttentionRUL (Seed 42)",
    "CNNBiLSTMAttentionRUL (Seed 123)",
    "CNNBiLSTMAttentionRUL (Seed 999)"
]

sub_seeds = df_official_eval[df_official_eval["Model"].isin(seed_model_names)]

stability_metrics = [
    ("window_mae_s", "Window MAE (s)"),
    ("window_rmse_s", "Window RMSE (s)"),
    ("endpoint_mae_s", "Endpoint MAE (s)"),
    ("endpoint_rmse_s", "Endpoint RMSE (s)"),
    ("ieee_phm_score", "IEEE PHM Score")
]

stability_records = []
for metric_key, col_name in stability_metrics:
    vals = sub_seeds[col_name].to_numpy()
    stability_records.append({
        "metric": metric_key,
        "display_name": col_name,
        "mean": round(float(np.mean(vals)), 5 if "score" in metric_key else 1),
        "sample_std": round(float(np.std(vals, ddof=1)), 5 if "score" in metric_key else 1),
        "min": round(float(np.min(vals)), 5 if "score" in metric_key else 1),
        "max": round(float(np.max(vals)), 5 if "score" in metric_key else 1),
        "seed_42_val": float(sub_seeds[sub_seeds["Model"] == "CNNBiLSTMAttentionRUL (Seed 42)"][col_name].iloc[0]),
        "seed_123_val": float(sub_seeds[sub_seeds["Model"] == "CNNBiLSTMAttentionRUL (Seed 123)"][col_name].iloc[0]),
        "seed_999_val": float(sub_seeds[sub_seeds["Model"] == "CNNBiLSTMAttentionRUL (Seed 999)"][col_name].iloc[0]),
        "std_convention": "sample_standard_deviation_ddof_1"
    })

df_seed_stability = pd.DataFrame(stability_records)
seed_stab_path = REPORTS_DIR / "official_seed_stability.csv"
df_seed_stability.to_csv(seed_stab_path, index=False)

print(f"Official seed stability statistics exported to: {to_portable_path(seed_stab_path)}")
print(df_seed_stability[["metric", "mean", "sample_std", "min", "max"]].to_string(index=False))


Official seed stability statistics exported to: <PROJECT_ROOT>/customer_handover/reports/official_seed_stability.csv
         metric        mean  sample_std        min         max
   window_mae_s  8329.50000   206.90000 8090.60000  8450.90000
  window_rmse_s 10083.90000   269.90000 9772.50000 10251.30000
 endpoint_mae_s  3358.10000   389.70000 2972.50000  3751.80000
endpoint_rmse_s  4213.50000   800.40000 3529.00000  5093.60000
 ieee_phm_score     0.10732     0.02272    0.08955     0.13292


### Compute and export operating-condition breakdown comparison


In [24]:
oc_records = []

# 1. Development Validation by Operating Condition
dev_cond_models = [
    ("Random Forest", df_rf),
    ("LateConcatBaseline", df_dl[df_dl["model_name"] == "LateConcatBaseline"]),
    ("VibrationOnlyBaseline", df_dl[df_dl["model_name"] == "VibrationOnlyBaseline"]),
    ("CNNBiLSTMAttentionRUL", df_dl[df_dl["model_name"] == "CNNBiLSTMAttentionRUL"])
]

for m_label, df_m in dev_cond_models:
    for cond in sorted(df_m["condition_id"].unique()):
        sub_c = df_m[df_m["condition_id"] == cond]
        ep = sub_c[sub_c["is_final_endpoint"]]
        e_actual = ep["actual_rul_seconds"].to_numpy()
        e_pred = ep["predicted_rul_seconds"].to_numpy()
        e_mae = float(np.mean(np.abs(e_actual - e_pred)))
        e_rmse = float(np.sqrt(np.mean((e_actual - e_pred) ** 2)))
        
        oc_records.append({
            "split": "development_validation",
            "model_name": m_label,
            "condition_id": cond,
            "num_windows": len(sub_c),
            "num_endpoints": len(ep),
            "endpoint_mae_s": round(e_mae, 1),
            "endpoint_rmse_s": round(e_rmse, 1),
            "ieee_phm_score": None
        })

# 2. Official Test by Operating Condition
test_cond_models = [
    ("Random Forest", df_official_rf),
    ("CNNBiLSTMAttentionRUL (Ensemble)", ensemble_dl)
]

for m_label, df_m in test_cond_models:
    for cond in sorted(df_m["condition_id"].unique()):
        sub_c = df_m[df_m["condition_id"] == cond]
        ep = sub_c[sub_c["is_final_endpoint"]]
        e_actual = ep["actual_rul_seconds"].to_numpy()
        e_pred = ep["predicted_rul_seconds"].to_numpy()
        e_mae = float(np.mean(np.abs(e_actual - e_pred)))
        e_rmse = float(np.sqrt(np.mean((e_actual - e_pred) ** 2)))
        phm = compute_canonical_ieee_phm_score(e_actual, e_pred)
        
        oc_records.append({
            "split": "official_test",
            "model_name": m_label,
            "condition_id": cond,
            "num_windows": len(sub_c),
            "num_endpoints": len(ep),
            "endpoint_mae_s": round(e_mae, 1),
            "endpoint_rmse_s": round(e_rmse, 1),
            "ieee_phm_score": round(phm, 5) if phm is not None else None
        })

df_oc = pd.DataFrame(oc_records)
oc_path = REPORTS_DIR / "operating_condition_comparison.csv"
df_oc.to_csv(oc_path, index=False)

df_oc_disp = df_oc.copy()
df_oc_disp["ieee_phm_score"] = df_oc_disp["ieee_phm_score"].apply(lambda v: "N/A" if v is None or pd.isna(v) else f"{v:.5f}")

print(f"Operating condition breakdown comparison exported to: {to_portable_path(oc_path)}")
print(df_oc_disp[["split", "model_name", "condition_id", "num_endpoints", "endpoint_mae_s", "endpoint_rmse_s", "ieee_phm_score"]].to_string(index=False))


Operating condition breakdown comparison exported to: <PROJECT_ROOT>/customer_handover/reports/operating_condition_comparison.csv
                 split                       model_name condition_id  num_endpoints  endpoint_mae_s  endpoint_rmse_s ieee_phm_score
development_validation                    Random Forest          OC1              2          1647.9           1851.5            N/A
development_validation                    Random Forest          OC2              2          4674.3           5044.5            N/A
development_validation                    Random Forest          OC3              2          1339.3           1339.3            N/A
development_validation               LateConcatBaseline          OC1              2          2200.9           3112.5            N/A
development_validation               LateConcatBaseline          OC2              2          1173.5           1177.4            N/A
development_validation               LateConcatBaseline          OC3          

### Plot per-bearing endpoint error visualization for official test set


In [25]:
plt.figure(figsize=(12, 6))

ep_rf = df_official_diag[df_official_diag["model_name"] == "Random Forest (Handcrafted)"].sort_values("bearing_id")
ep_ens = df_official_diag[df_official_diag["model_name"] == "CNNBiLSTMAttentionRUL (Multi-Seed Ensemble)"].sort_values("bearing_id")

bearings = ep_rf["bearing_id"].tolist()
x = np.arange(len(bearings))
width = 0.35

rf_signed_err = ep_rf["signed_error_seconds"].tolist()
ens_signed_err = ep_ens["signed_error_seconds"].tolist()

plt.bar(x - width/2, rf_signed_err, width, label="Random Forest Baseline", color="navy", alpha=0.85)
plt.bar(x + width/2, ens_signed_err, width, label="CNNBiLSTMAttentionRUL (Ensemble)", color="crimson", alpha=0.85)

plt.axhline(0, color="black", linestyle="--", linewidth=1.0)
plt.xlabel("Bearing Identifier (Official Test Set)")
plt.ylabel("Signed Endpoint Error (seconds) [Predicted - Actual]")
plt.title("Official Test Set: Per-Bearing Signed Endpoint Error (11 Bearings)")
plt.xticks(x, bearings, rotation=30, ha="right")
plt.legend()
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

per_bearing_plot_path = PLOTS_DIR / "official_per_bearing_error.png"
plt.savefig(per_bearing_plot_path, dpi=150)
plt.close()

print(f"Per-bearing error plot exported to: {to_portable_path(per_bearing_plot_path)}")


Per-bearing error plot exported to: <PROJECT_ROOT>/customer_handover/reports/plots/official_per_bearing_error.png


### Compile and export compact objective status assessment JSON


In [26]:
objective_status = {
    "evaluation_version": "2.1.0-step21d",
    "assessment_date": "2026-10-24",
    "status_summary": {
        "achieved": 2,
        "partially_achieved": 2,
        "not_achieved": 3,
        "not_yet_evaluated": 1
    },
    "objectives": [
        {
            "objective_id": "OBJ_01",
            "name": "End-to-End Multimodal RUL Regression Pipeline",
            "status": "Achieved",
            "evidence": "Complete reproducible pipeline with clean data gate, dual checkpoints, and 14-column prediction tables across 6 folds and official test.",
            "source": "SPEC.md Section 1, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_02",
            "name": "Independent Official Test Evaluation Without Leakage",
            "status": "Achieved",
            "evidence": "11 official test bearings evaluated strictly once per frozen seed with weights frozen; non-analytic data-gate verification only.",
            "source": "SPEC.md Section 5, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_03",
            "name": "Missing-Temperature Availability Gating and Robustness",
            "status": "Partially Achieved",
            "evidence": "Architecture handles missing temperature without NaN/Inf via availability gating; operational on all 3 missing-temperature test bearings. However, temperature has not demonstrated superior accuracy over vibration-only modeling.",
            "source": "SPEC.md Section 8, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_04",
            "name": "Outperform Traditional Handcrafted Machine Learning Baseline",
            "status": "Partially Achieved",
            "evidence": "Multi-seed ensemble slightly improves official endpoint MAE (3264.0s vs 3502.9s) and RMSE (4140.0s vs 4378.0s), but Random Forest achieves superior window MAE (6558.1s vs 8191.5s) and higher IEEE PHM score (0.11422 vs 0.08465).",
            "source": "SPEC.md Section 6.2, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_05",
            "name": "Outperform Single-Modality Deep Learning Baselines",
            "status": "Not Achieved",
            "evidence": "On 6-fold development validation, VibrationOnlyBaseline achieves 1738.0s Endpoint MAE and 1947.8s Endpoint RMSE, outperforming the multimodal CNN-BiLSTM-Attention model (2979.8s Endpoint MAE, 3190.2s Endpoint RMSE).",
            "source": "SPEC.md Section 6.3, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_06",
            "name": "Demonstrate Cross-Modal and Temporal Attention Contribution",
            "status": "Not Achieved",
            "evidence": "MainModel_NoAttention achieves 2552.0s Endpoint MAE on 6-fold development validation compared to 2979.8s for CNNBiLSTMAttentionRUL. Current ablation evidence indicates temporal attention adds optimization difficulty rather than benefit.",
            "source": "SPEC.md Section 7.4 & 12, FA26AI57 Section 3"
        },
        {
            "objective_id": "OBJ_07",
            "name": "Achieve Lifetime RMSE Below 10%",
            "status": "Not Achieved",
            "evidence": "Official test endpoint RMSE is 4,140.0s (relative RMSE ~20-23% of operating duration). On 6-fold development validation, the winning candidate (OPT2_06) achieves a normalized trajectory RMSE of 0.288 (28.8% of bearing lifetime). Both metrics exceed the 10% target. Per SPEC Section 20, development endpoint error with zero target cannot be used to claim this objective.",
            "source": "FA26AI57 Section 3, SPEC.md Section 20"
        },
        {
            "objective_id": "OBJ_08",
            "name": "Transformer Architecture Comparison",
            "status": "Not Yet Evaluated",
            "evidence": "Deferred until development optimization of primary CNN-BiLSTM architecture is complete in accordance with SPEC.md Section 7.4.",
            "source": "SPEC.md Section 7.4, FA26AI57 Section 3"
        }
    ]
}

obj_status_path = REPORTS_DIR / "objective_status.json"
with open(obj_status_path, "w", encoding="utf-8") as f:
    json.dump(objective_status, f, indent=2)

print(f"Objective status summary exported to: {to_portable_path(obj_status_path)}")
print(f"Status Summary: {objective_status['status_summary']}")


Objective status summary exported to: <PROJECT_ROOT>/customer_handover/reports/objective_status.json
Status Summary: {'achieved': 2, 'partially_achieved': 2, 'not_achieved': 3, 'not_yet_evaluated': 1}


### Verify SPEC Section 20.1 audit metric fixture on preserved OPT_06 predictions


In [27]:
print("=== SPEC Section 20.1 Audit Metric Fixture Verification ===")
opt_preds_path = PREDICTIONS_DIR / "optimization_dev_predictions.csv"
if opt_preds_path.exists():
    df_opt = pd.read_csv(opt_preds_path)
    opt06 = df_opt[df_opt["run_id"].str.contains("OPT_06")]
    
    act_win = opt06["actual_rul_seconds"].to_numpy()
    pred_win = opt06["predicted_rul_seconds"].to_numpy()
    win_mae = float(np.mean(np.abs(pred_win - act_win)))
    win_rmse = float(np.sqrt(np.mean((pred_win - act_win) ** 2)))
    
    opt06_ep = opt06[opt06["is_final_endpoint"]]
    act_ep = opt06_ep["actual_rul_seconds"].to_numpy()
    pred_ep = opt06_ep["predicted_rul_seconds"].to_numpy()
    ep_mae = float(np.mean(np.abs(pred_ep - act_ep)))
    ep_rmse = float(np.sqrt(np.mean((pred_ep - act_ep) ** 2)))
    
    print(f"OPT_06 Endpoint MAE:         {ep_mae:.4f}s (Expected ~359.9s)")
    print(f"OPT_06 Global Endpoint RMSE: {ep_rmse:.4f}s (Expected ~476.8s)")
    print(f"OPT_06 Window MAE:           {win_mae:.4f}s (Expected ~4279.7s)")
    print(f"OPT_06 Global Window RMSE:   {win_rmse:.4f}s (Expected ~6486.8s)")
    
    assert abs(ep_mae - 359.8962) < 0.2, f"Endpoint MAE mismatch: {ep_mae}"
    assert abs(ep_rmse - 476.7934) < 0.2, f"Global Endpoint RMSE mismatch: {ep_rmse}"
    assert abs(win_mae - 4279.7167) < 0.5, f"Window MAE mismatch: {win_mae}"
    assert abs(win_rmse - 6486.8132) < 0.5, f"Global Window RMSE mismatch: {win_rmse}"
    print("SPEC Section 20.1 Audit Metric Fixture: ALL PASS")


=== SPEC Section 20.1 Audit Metric Fixture Verification ===
OPT_06 Endpoint MAE:         359.8962s (Expected ~359.9s)
OPT_06 Global Endpoint RMSE: 476.7934s (Expected ~476.8s)
OPT_06 Window MAE:           4279.7167s (Expected ~4279.7s)
OPT_06 Global Window RMSE:   6486.8132s (Expected ~6486.8s)
SPEC Section 20.1 Audit Metric Fixture: ALL PASS


### Execute three-way independent metric verification and prediction artifact auditing


In [28]:
import datetime
import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd

print("=== SPEC Section 23.1 Independent Metric Verification & Generator Provenance ===")
v2_preds_path = PREDICTIONS_DIR / "optimization_v2_dev_predictions.csv"
sel_v2_path = REPORTS_DIR / "optimized_model_selection_v2.json"
metric_verif_path = REPORTS_DIR / "optimization_metric_verification_v2.json"

assert v2_preds_path.exists(), f"Missing {v2_preds_path}"
assert sel_v2_path.exists(), f"Missing {sel_v2_path}"

# A. Prediction Artifact Auditing
with open(v2_preds_path, "rb") as f:
    csv_bytes = f.read()
csv_sha256 = hashlib.sha256(csv_bytes).hexdigest()
csv_size = len(csv_bytes)

df_opt2 = pd.read_csv(v2_preds_path)
cols = list(df_opt2.columns)
schema_sha = hashlib.sha256(json.dumps(cols).encode("utf-8")).hexdigest()

REQUIRED_COLS = [
    "run_id", "model_name", "fold", "split", "seed", "bearing_id",
    "condition_id", "endpoint_acquisition_index", "endpoint_timestamp",
    "actual_rul_seconds", "predicted_rul_seconds", "absolute_error_seconds",
    "is_temperature_fully_missing", "is_final_endpoint"
]
schema_verdict = "PASS" if all(c in cols for c in REQUIRED_COLS) else "FAIL"
assert schema_verdict == "PASS", f"Prediction schema missing required columns!"

pred_identity = {
    "relative_path": "predictions/optimization_v2_dev_predictions.csv",
    "portable_path": "<PROJECT_ROOT>/customer_handover/predictions/optimization_v2_dev_predictions.csv",
    "sha256": csv_sha256,
    "byte_size": csv_size,
    "row_count": len(df_opt2),
    "endpoint_count": int(df_opt2["is_final_endpoint"].sum()),
    "columns": cols,
    "schema_sha256": schema_sha,
    "schema_verdict": schema_verdict
}

# B. Selection Manifest Identity
with open(sel_v2_path, "rb") as f:
    sel_bytes = f.read()
sel_sha256 = hashlib.sha256(sel_bytes).hexdigest()
sel_size = len(sel_bytes)
sel_data = json.loads(sel_bytes.decode("utf-8"))

sel_identity = {
    "relative_path": "reports/optimized_model_selection_v2.json",
    "portable_path": "<PROJECT_ROOT>/customer_handover/reports/optimized_model_selection_v2.json",
    "sha256": sel_sha256,
    "byte_size": sel_size
}

sel_finalists = {r["candidate_id"]: r for r in sel_data.get("finalist_confirmations", [])}
assert len(sel_finalists) == 3, f"Expected 3 confirmed finalists in selection report, found {len(sel_finalists)}"

# C. Authentic Generator Provenance (SPEC 24.1)
CELL_ID = "cell_28_independent_metric_verification"
VERIF_CODE_VERSION = "step21h"
NORMALIZATION_METHOD = "join_source_normalize_lf"
execution_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()

# Locate this unique cell in saved 04 notebook JSON
nb_04_file_path = BASE_DIR / "04_evaluation_and_ablation.ipynb"
with open(nb_04_file_path, "r", encoding="utf-8") as f:
    nb_04_json = json.load(f)

matching_cells = [
    c for c in nb_04_json.get("cells", [])
    if c.get("metadata", {}).get("id") == CELL_ID or 
       (c.get("cell_type") == "code" and f'CELL_ID = "{CELL_ID}"' in "".join(c.get("source", [])))
]
assert len(matching_cells) == 1, f"Expected exactly 1 cell with ID {CELL_ID}, found {len(matching_cells)}"
target_cell = matching_cells[0]
source_lines = target_cell.get("source", [])
assert any(f'VERIF_CODE_VERSION = "{VERIF_CODE_VERSION}"' in line for line in source_lines),     f"Expected version marker {VERIF_CODE_VERSION} not found in saved cell source!"

# Normalization procedure: join raw source array, normalize CRLF to LF
normalized_cell_source = "".join(source_lines).replace(chr(13) + chr(10), chr(10))
cell_source_hash = hashlib.sha256(normalized_cell_source.encode("utf-8")).hexdigest()

with open(nb_04_file_path, "rb") as f:
    delivered_nb_sha256 = hashlib.sha256(f.read()).hexdigest()

generator_metadata = {
    "notebook_path": "<PROJECT_ROOT>/customer_handover/04_evaluation_and_ablation.ipynb",
    "cell_id": CELL_ID,
    "verification_code_version": VERIF_CODE_VERSION,
    "normalization_method": NORMALIZATION_METHOD,
    "cell_source_hash": cell_source_hash,
    "delivered_notebook_file_sha256": delivered_nb_sha256,
    "execution_timestamp": execution_timestamp
}

# D. Define Fixed Section 21.1 Audit Fixtures and Tolerances
SECTION_21_FIXTURES = {
    "OPT2_06": {
        "mean_trajectory_nrmse": 0.28789,
        "mean_trajectory_nmae": 0.22602,
        "global_window_mae_s": 4279.7,
        "global_window_rmse_s": 6486.8,
        "global_endpoint_mae_s": 359.9,
        "global_endpoint_rmse_s": 476.8
    },
    "OPT2_05": {
        "mean_trajectory_nrmse": 0.31834,
        "mean_trajectory_nmae": 0.27172,
        "global_window_mae_s": 4740.2,
        "global_window_rmse_s": 6855.1,
        "global_endpoint_mae_s": 1459.2,
        "global_endpoint_rmse_s": 1600.3
    },
    "OPT2_09": {
        "mean_trajectory_nrmse": 0.34792,
        "mean_trajectory_nmae": 0.29596,
        "global_window_mae_s": 4936.6,
        "global_window_rmse_s": 6714.7,
        "global_endpoint_mae_s": 3159.3,
        "global_endpoint_rmse_s": 3703.6
    }
}

TOLERANCES = {
    "mean_trajectory_nrmse": 0.0001,
    "mean_trajectory_nmae": 0.0001,
    "global_window_mae_s": 0.2,
    "global_window_rmse_s": 0.2,
    "global_endpoint_mae_s": 0.2,
    "global_endpoint_rmse_s": 0.2
}

finalists = ["OPT2_06", "OPT2_05", "OPT2_09"]
candidate_evals = {}
all_finalists_passed = True

for cand_id in finalists:
    cand_df = df_opt2[df_opt2["run_id"].str.contains(cand_id)]
    assert len(cand_df) == 468, f"Expected 468 rows for {cand_id}, got {len(cand_df)}"
    
    b_nrmses, b_nmaes = [], []
    bearing_records = []
    for b_id, grp in cand_df.groupby("bearing_id"):
        act = grp["actual_rul_seconds"].to_numpy()
        pred = grp["predicted_rul_seconds"].to_numpy()
        scale = float(np.max(act))
        assert scale > 0, f"Bearing {b_id} scale must be positive"
        mae = float(np.mean(np.abs(pred - act)))
        rmse = float(np.sqrt(np.mean((pred - act) ** 2)))
        nmae = mae / scale
        nrmse = rmse / scale
        b_nmaes.append(nmae)
        b_nrmses.append(nrmse)
        bearing_records.append({
            "bearing_id": b_id,
            "window_count": len(grp),
            "lifetime_scale_s": scale,
            "trajectory_mae_s": round(mae, 1),
            "trajectory_rmse_s": round(rmse, 1),
            "trajectory_nmae": round(nmae, 5),
            "trajectory_nrmse": round(nrmse, 5)
        })
        
    all_act = cand_df["actual_rul_seconds"].to_numpy()
    all_pred = cand_df["predicted_rul_seconds"].to_numpy()
    g_win_mae = float(np.mean(np.abs(all_pred - all_act)))
    g_win_rmse = float(np.sqrt(np.mean((all_pred - all_act) ** 2)))
    
    ep_df = cand_df[cand_df["is_final_endpoint"]]
    assert len(ep_df) == 6, f"Expected 6 endpoints for {cand_id}, got {len(ep_df)}"
    ep_act = ep_df["actual_rul_seconds"].to_numpy()
    ep_pred = ep_df["predicted_rul_seconds"].to_numpy()
    g_ep_mae = float(np.mean(np.abs(ep_pred - ep_act)))
    g_ep_rmse = float(np.sqrt(np.mean((ep_pred - ep_act) ** 2)))
    
    recomp = {
        "mean_trajectory_nrmse": round(float(np.mean(b_nrmses)), 5),
        "mean_trajectory_nmae": round(float(np.mean(b_nmaes)), 5),
        "global_window_mae_s": round(g_win_mae, 1),
        "global_window_rmse_s": round(g_win_rmse, 1),
        "global_endpoint_mae_s": round(g_ep_mae, 1),
        "global_endpoint_rmse_s": round(g_ep_rmse, 1)
    }
    
    sel_rec = sel_finalists[cand_id]
    sel_metrics = {
        "mean_trajectory_nrmse": sel_rec["mean_trajectory_nrmse"],
        "mean_trajectory_nmae": sel_rec["mean_trajectory_nmae"],
        "global_window_mae_s": sel_rec["global_window_mae_s"],
        "global_window_rmse_s": sel_rec["global_window_rmse_s"],
        "global_endpoint_mae_s": sel_rec["global_endpoint_mae_s"],
        "global_endpoint_rmse_s": sel_rec["global_endpoint_rmse_s"]
    }
    
    target_fixture = SECTION_21_FIXTURES[cand_id]
    
    # Comparison 1: Recomputed vs Selection Report
    diff_sel = {k: round(abs(recomp[k] - sel_metrics[k]), 5) for k in recomp}
    verdicts_sel = {k: "PASS" if diff_sel[k] <= TOLERANCES[k] else "FAIL" for k in recomp}
    
    # Comparison 2: Recomputed vs Fixed Fixture
    diff_fix = {k: round(abs(recomp[k] - target_fixture[k]), 5) for k in recomp}
    verdicts_fix = {k: "PASS" if diff_fix[k] <= TOLERANCES[k] else "FAIL" for k in recomp}
    
    c_pass = all(v == "PASS" for v in verdicts_sel.values()) and all(v == "PASS" for v in verdicts_fix.values())
    if not c_pass:
        all_finalists_passed = False
        print(f"FAILED verification for {cand_id}:")
        print(f"  diff_sel: {diff_sel}")
        print(f"  diff_fix: {diff_fix}")
        
    candidate_evals[cand_id] = {
        "candidate_id": cand_id,
        "window_row_count": len(cand_df),
        "endpoint_row_count": len(ep_df),
        "recomputed_metrics": recomp,
        "selection_report_metrics": sel_metrics,
        "fixture_metrics": target_fixture,
        "absolute_differences_vs_selection": diff_sel,
        "absolute_differences_vs_fixture": diff_fix,
        "tolerances": TOLERANCES,
        "verdicts_vs_selection": verdicts_sel,
        "verdicts_vs_fixture": verdicts_fix,
        "verdict": "PASS" if c_pass else "FAIL",
        "bearing_breakdown": bearing_records
    }
    print(f"Finalist {cand_id}: Recomputed vs Selection Report & Fixture -> {'PASS' if c_pass else 'FAIL'}")

overall_verif_status = "PASS" if all_finalists_passed else "FAIL"
metric_verif_report = {
    "report_name": "optimization_metric_verification_v2",
    "generator": generator_metadata,
    "timestamp": execution_timestamp,
    "status": overall_verif_status,
    "input_artifacts": {
        "prediction_artifact": pred_identity,
        "selection_manifest": sel_identity
    },
    "prediction_artifact_identity": pred_identity,
    "selection_manifest_path": "<PROJECT_ROOT>/customer_handover/reports/optimized_model_selection_v2.json",
    "evaluated_finalists": finalists,
    "tolerances": TOLERANCES,
    "candidate_evaluations": candidate_evals
}

with open(metric_verif_path, "w", encoding="utf-8") as f:
    json.dump(metric_verif_report, f, indent=2)

print(f"Generated Report Path: {to_portable_path(metric_verif_path)}")
print(f"Saved Cell Source SHA-256: {cell_source_hash}")
print(f"Prediction Artifact SHA-256: {csv_sha256}")
print(f"Selection Manifest SHA-256: {sel_sha256}")
print(f"Overall Metric Verification Verdict: {overall_verif_status}")
assert overall_verif_status == "PASS", "Independent metric verification failed!"


=== SPEC Section 23.1 Independent Metric Verification & Generator Provenance ===
Finalist OPT2_06: Recomputed vs Selection Report & Fixture -> PASS
Finalist OPT2_05: Recomputed vs Selection Report & Fixture -> PASS
Finalist OPT2_09: Recomputed vs Selection Report & Fixture -> PASS
Generated Report Path: <PROJECT_ROOT>/customer_handover/reports/optimization_metric_verification_v2.json
Saved Cell Source SHA-256: 038385d694691ab302a90026eb068fd4b836e15a8e99cb17c0cc644cb0cec74e
Prediction Artifact SHA-256: 937250df4bafcb717edec805c6fa034a76142b43b0382b940b9477e0b78323f8
Selection Manifest SHA-256: be022abf8ad14c5bc138f64c0ab33da88999cb71ed2e9ba66648fb76c4ae6ebf
Overall Metric Verification Verdict: PASS
